# Extending Constrained Belief (Kaggle)

Trains a Mess3 / Linear_Mess3 model. See `README.md` in the repo for the
full config reference this notebook mirrors.

Before running:
- **Edit `repo_url` in the clone cell below** to point at wherever this
  repo actually lives on GitHub.
- Add a `GITHUB_TOKEN` Kaggle secret if the repo isn't public.
- Add a `WANDB_API_KEY` Kaggle secret, or leave `logging_config.wandb = False`
  below to run without wandb.

In [ ]:
!pip install -q "pyarrow==15.0.2" "datasets==2.18.0" "transformer-lens==1.14.0"

In [ ]:
!pip install -q "datashader"

In [ ]:
!pip install -q cupy-cuda12x

### Clone the repo

In [ ]:
import os

# Provide GITHUB_TOKEN via Kaggle Secrets (Add-ons > Secrets) if the repo
# isn't public. Nothing is hardcoded here.
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN")

# EDIT ME: point this at wherever the repo actually lives.
repo_url = "github.com/<YOUR_GITHUB_USERNAME>/extending-constrained-belief"
clone_url = f"https://{github_token}@{repo_url}" if github_token else f"https://{repo_url}"

!git clone {clone_url}

### Install as an editable package

In [ ]:
%cd /kaggle/working/extending-constrained-belief
!pip install -q -e .

In [ ]:
%cd /kaggle/working/extending-constrained-belief/epsilon_transformers

In [ ]:
from epsilon_transformers.training.configs.model_configs import RawModelConfig
from epsilon_transformers.training.configs.training_configs import (
    LoggingConfig,
    OptimizerConfig,
    PersistanceConfig,
    ProcessDatasetConfig,
    TrainConfig,
    NGramAnalysisConfig,
    MarkovKLAnalysisConfig,
    SimplexAnalysisConfig,
    AnalysisConfig,
)
from epsilon_transformers.training.train import train_model

### Model Configuration

Identical across every reference run — see `README.md`.

In [ ]:
model_config = RawModelConfig(
    d_vocab=3,
    d_model=64,
    n_ctx=10,
    d_head=8,
    n_head=2,
    d_mlp=256,
    n_layers=1,
)

### Optimizer Configuration

In [ ]:
optimizer_config = OptimizerConfig(
    optimizer_type="adam",
    learning_rate=0.0001,
    weight_decay=0,
)

### Dataset Configuration

Only `process` / `process_params` differed between reference runs — pick
`"Mess3"` or `"Linear_Mess3"` and the `x`/`a` values you want to train.
`mixing`/`mixing_params` don't exist on `ProcessDatasetConfig` (the
switching-process feature isn't part of this repo), so don't pass them
here.

In [ ]:
dataset_config = ProcessDatasetConfig(
    process="Mess3",
    process_params={"x": 0.28, "a": 0.85},
    batch_size=128,
    num_tokens=2000000000,
    sequence_length=10,
    test_split=0.0005,
    chunk_size=2048,
    test_batch_size=1000000,
    start_state_idx=None,  # let the process compute its own steady state
)

In [ ]:
path = "0.28_0.85"  # used for the run name / output folder below

### Persistence Configuration

In [ ]:
from pathlib import Path

persistance_config = PersistanceConfig(
    location="local",
    collection_location=Path(f"models/{path}"),
    checkpoint_every_n_tokens=3000000,
)

### Logging Configuration

Reads `WANDB_API_KEY` from Kaggle Secrets (falling back to the
environment); no key is hardcoded. Set `wandb=False` to skip wandb
entirely.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    wandb_api_key = os.environ.get("WANDB_API_KEY")

logging_config = LoggingConfig(
    project_name="epstrans",
    wandb=wandb_api_key is not None,
    wandb_api_key=wandb_api_key,
    run_name=path,
    relative_loss=False,
)

### Analysis Configuration

In [ ]:
analysis_config = AnalysisConfig(
    analysis_batch_size=100,
    ngram_analysis=NGramAnalysisConfig(
        enabled=False,
        n_values=[1, 2, 3],
        return_per_position=False,
    ),
    markov_kl_analysis=MarkovKLAnalysisConfig(
        enabled=True,
        return_per_position=False,
    ),
    simplex_analysis=SimplexAnalysisConfig(
        enabled=True,
        hook_point="blocks.0.hook_resid_post",
        num_samples_for_probe=None,
    ),
)

### Complete Training Configuration

In [ ]:
mock_config = TrainConfig(
    model=model_config,
    optimizer=optimizer_config,
    dataset=dataset_config,
    persistance=persistance_config,
    logging=logging_config,
    analysis=analysis_config,
    verbose=True,
    seed=42,
    do_eval=True,
)

### Main Entry Point

At `num_tokens=2_000_000_000` (the reference scale), this takes **around 9
hours on a Kaggle T4 GPU**.

In [ ]:
if __name__ == "__main__":
    try:
        train_model(mock_config)
    except ValueError as e:
        print(f"Configuration Error: {e}")
        print("\nFix: Either:")
        print("  1. Set wandb_api_key in LoggingConfig")
        print("  2. OR set WANDB_API_KEY environment variable / Kaggle secret")
        raise

### Archive checkpoints for download

In [ ]:
import shutil
import os

folder_to_zip = f"/kaggle/working/extending-constrained-belief/epsilon_transformers/models/{path}"
output_filename = f"chckpts_archive_{path}"

shutil.make_archive(output_filename, "zip", root_dir="/kaggle/working/", base_dir=folder_to_zip)

print(f"Successfully created {output_filename}.zip")

### Selective archive: last checkpoint + logs + config

In [ ]:
import shutil
import os
import re
from pathlib import Path

model_dir = Path(f"/kaggle/working/extending-constrained-belief/epsilon_transformers/models/{path}")
output_filename = f"selective_archive_{path}"
tmp_stage_dir = Path(f"/kaggle/working/stage_{path}")
tmp_stage_dir.mkdir(parents=True, exist_ok=True)

def parse_tokens(p: Path) -> int:
    short = re.match(r"^(\d+)\.pt$", p.name)
    if short:
        return int(short.group(1))
    long = re.search(r"tokens_(\d+)", p.name)
    if long:
        return int(long.group(1))
    return -1

all_checkpoints = [
    p for p in model_dir.glob("*.pt")
    if "ngram_counts" not in p.name and parse_tokens(p) != -1
]

if not all_checkpoints:
    raise FileNotFoundError(f"No checkpoints found in {model_dir}")

last_checkpoint = sorted(all_checkpoints, key=parse_tokens)[-1]
print(f"Last checkpoint: {last_checkpoint.name}")

files_to_archive = {
    last_checkpoint.name: last_checkpoint,
    "train_logs.csv": model_dir / "train_logs.csv",
    "test_logs.csv": model_dir / "test_logs.csv",
    "train_config.json": model_dir / "train_config.json",
}

copied = []
for dest_name, src_path in files_to_archive.items():
    if src_path.exists():
        shutil.copy2(src_path, tmp_stage_dir / dest_name)
        copied.append(dest_name)
        print(f"  Staged: {dest_name}")
    else:
        print(f"  Skipped (not found): {dest_name}")

archive_path = shutil.make_archive(
    output_filename, "zip",
    root_dir=tmp_stage_dir.parent,
    base_dir=tmp_stage_dir.name,
)

shutil.rmtree(tmp_stage_dir)

print(f"\nArchive created: {archive_path}")
print(f"Included files: {copied}")